# Four Times the Batch, Half the Noise

**Topic:** Minibatch gradient noise &nbsp;·&nbsp; **Difficulty:** Medium

Minibatch gradients are unbiased estimates of the full gradient, and their
standard deviation falls as `1/sqrt(B)`. That single fact governs almost every
practical scaling decision: it is why doubling the batch does not halve the
noise, why huge batches hit diminishing returns, and why the linear-scaling rule
for learning rate exists.

Measure the curve directly, and check the exponent.

## The data (created in the next cell)

- `X` `(20000, 30)`, `y` `(20000,)` for a least-squares objective.
- `w0`: the point at which gradients are measured.
- `batch_sizes = [1, 4, 16, 64, 256, 1024, 4096]`.

## Your task

With **NumPy only**:

1. Compute the exact full-batch gradient `g_full` of the mean squared error at
   `w0`.
2. For each batch size `B`, draw 300 independent minibatches (sampling **with**
   replacement, `np.random.default_rng(4000 + B)`), compute each minibatch
   gradient, and record the mean L2 distance from `g_full`, calling it
   `noise[B]`.
3. Fit `log(noise) = a + b * log(B)` by least squares and report the slope `b`.
4. Set `answers` to a dict with `"noise"` (length-7), `"slope"`, `"bias_check"`:
   the L2 norm of (mean minibatch gradient at `B = 64` minus `g_full`), which
   should be near zero.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks the fitted slope is close to `-0.5` and that the estimator is unbiased.

In [ ]:
import numpy as np

rng = np.random.default_rng(3006)
n, d = 20000, 30
X = rng.normal(0, 1, (n, d))
w_star = rng.normal(0, 1, d)
y = X @ w_star + rng.normal(0, 2.0, n)
w0 = w_star + rng.normal(0, 0.5, d)
batch_sizes = [1, 4, 16, 64, 256, 1024, 4096]
print(f"X {X.shape}, measuring gradient noise at a fixed point w0")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("noise", [7], 61.93458058549363, -4.84088935383495, 1e-06),
    ("slope", [], -0.48558114382304496, 0.08130474275308446, 1e-06),
    ("bias_check", [], 0.19694687054668292, -0.032976393028264744, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "slope" in answers:
    _s = abs(answers["slope"] + 0.5) < 0.03
    _u = answers["bias_check"] < 0.15 * answers["noise"][0]
    print(f"  {'slope is -1/2':<24} {'ok' if _s else 'wrong'}  (slope = {answers['slope']:.4f})")
    print(f"  {'estimator unbiased':<24} {'ok' if _u else 'wrong'}")
    _ok = _ok and _s and _u

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

Each minibatch gradient is an average of `B` i.i.d. per-example gradients, so its
covariance is the per-example covariance divided by `B` and the standard
deviation falls like `B^(-1/2)`. Fitting the log-log slope recovers `-0.5` to
within sampling error. The practical reading: going from batch 32 to batch 1024
(32x the compute per step) buys only a 5.7x reduction in gradient noise, which is
why large-batch training needs a larger learning rate to make the extra
information count.

```python
import numpy as np

n = len(y)

def grad(idx, w):
    Xb, yb = X[idx], y[idx]
    r = Xb @ w - yb
    return 2.0 * (Xb.T @ r) / len(idx)

g_full = grad(np.arange(n), w0)

noise = []
bias_check = None
for B in batch_sizes:
    r = np.random.default_rng(4000 + B)
    gs = np.stack([grad(r.integers(0, n, B), w0) for _ in range(300)])
    noise.append(float(np.linalg.norm(gs - g_full, axis=1).mean()))
    if B == 64:
        bias_check = float(np.linalg.norm(gs.mean(axis=0) - g_full))

noise = np.array(noise)
logB = np.log(np.array(batch_sizes, dtype=float))
A = np.column_stack([np.ones_like(logB), logB])
coef, *_ = np.linalg.lstsq(A, np.log(noise), rcond=None)

answers = {"noise": noise, "slope": float(coef[1]),
           "bias_check": float(bias_check)}
```